# Campaign API exploration

Queries the provided campaign API so we can see response shapes before wiring the worker/frontend.

**Requires:** API running (`docker compose up api`) and `httpx` (`pip install httpx` or use the worker venv).

In [1]:
import json

import httpx

BASE_URL = "http://localhost:8080"
API_KEY = "interview-key-2024"

HEADERS = {"X-API-Key": API_KEY}


def get_json(path: str, params: dict | None = None) -> dict | list:
    url = f"{BASE_URL}{path}"
    response = httpx.get(url, headers=HEADERS, params=params, timeout=10.0)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    response.raise_for_status()
    return response.json()


def show(data) -> None:
    print(json.dumps(data, indent=2))

## GET /campaigns

Paginated list. Try `page`, `page_size` (max 10), and optional `status` (`active` | `paused` | `completed`).

In [5]:
campaigns_page = get_json("/campaigns", params={"page": 1, "page_size": 10})
show(campaigns_page)

GET http://localhost:8080/campaigns?page=1&page_size=10 -> 200
{
  "page": 1,
  "page_size": 10,
  "total": 50,
  "campaigns": [
    {
      "id": "3b1155c3-260b-87f9-75d3-4a4d5a71f6e0",
      "name": "Conditions Awareness - Migraine",
      "advertiser": "Lumen Pharma",
      "status": "active",
      "budget": 346643.68,
      "spend": 248527.12,
      "start_date": "2024-02-07",
      "end_date": "2024-05-29",
      "impressions": 1228497,
      "clicks": 21645,
      "cpm": 18.65
    },
    {
      "id": "e863e95a-9869-5017-5c00-9490056eb1b8",
      "name": "Q1 Brand Awareness - Cardiology",
      "advertiser": "Polaris Cardio",
      "status": "active",
      "budget": 116889.29,
      "spend": 92592.34,
      "start_date": "2024-02-20",
      "end_date": "2024-05-17",
      "impressions": 733408,
      "clicks": 3731,
      "cpm": 8.04
    },
    {
      "id": "c7c81db9-6bae-059b-2c17-73c6be116408",
      "name": "Patient Education - Rare Disease",
      "advertiser": "Lumen Phar

In [6]:
# Inspect top-level keys and first campaign record shape

if isinstance(campaigns_page, dict):
    print("Top-level keys:", list(campaigns_page.keys()))
    items = campaigns_page.get("campaigns") or campaigns_page.get("items") or []
else:
    items = campaigns_page

print(f"Items on this page: {len(items)}")

if items:
    first = items[0]
    print("\nFirst campaign keys:", list(first.keys()))
    show(first)
else:
    print("No campaigns returned")

Top-level keys: ['page', 'page_size', 'total', 'campaigns']
Items on this page: 10

First campaign keys: ['id', 'name', 'advertiser', 'status', 'budget', 'spend', 'start_date', 'end_date', 'impressions', 'clicks', 'cpm']
{
  "id": "3b1155c3-260b-87f9-75d3-4a4d5a71f6e0",
  "name": "Conditions Awareness - Migraine",
  "advertiser": "Lumen Pharma",
  "status": "active",
  "budget": 346643.68,
  "spend": 248527.12,
  "start_date": "2024-02-07",
  "end_date": "2024-05-29",
  "impressions": 1228497,
  "clicks": 21645,
  "cpm": 18.65
}


## GET /campaigns/{id}

Uses the first campaign id from the list above.

In [7]:
if not items:
    raise ValueError("Run the /campaigns cells first — no campaign id to fetch")

campaign_id = first["id"]
campaign = get_json(f"/campaigns/{campaign_id}")
show(campaign)

GET http://localhost:8080/campaigns/3b1155c3-260b-87f9-75d3-4a4d5a71f6e0 -> 200
{
  "id": "3b1155c3-260b-87f9-75d3-4a4d5a71f6e0",
  "name": "Conditions Awareness - Migraine",
  "advertiser": "Lumen Pharma",
  "status": "active",
  "budget": 346643.68,
  "spend": 248527.12,
  "start_date": "2024-02-07",
  "end_date": "2024-05-29",
  "impressions": 1228497,
  "clicks": 21645,
  "cpm": 18.65
}


## Optional: walk all pages

Uncomment/run if you want to see total count and every campaign id.

In [ ]:
page = 1
all_campaigns = []

while True:
    data = get_json("/campaigns", params={"page": page, "page_size": 10})
    batch = data.get("campaigns", data.get("items", [])) if isinstance(data, dict) else data
    if not batch:
        break
    all_campaigns.extend(batch)
    total = data.get("total") if isinstance(data, dict) else len(all_campaigns)
    print(f"page {page}: +{len(batch)} (running total {len(all_campaigns)}, api total={total})")
    if len(all_campaigns) >= (total or 0):
        break
    page += 1

print(f"\nFetched {len(all_campaigns)} campaigns")
for c in all_campaigns:
    print(c.get("id"), c.get("name"), c.get("status"), c.get("spend"), c.get("budget"))